# UTS Intelegensi Buatan - Kasus 10: Penumpang Titanic
Nama: Rizal Fahatrik Nugraha | NIM: 23552011452 | Kelas: TIF RP 23D

Jalankan sel dari atas ke bawah.

## 1. Siapkan folder & tulis file proyek

In [ ]:
import os
os.makedirs('dataset', exist_ok=True)
os.makedirs('models', exist_ok=True)
os.makedirs('notebook', exist_ok=True)

In [ ]:
%%writefile fitur.py
"""fitur.py - Fungsi feature engineering untuk Kasus 10 (Titanic).

Semua fungsi di sini bersifat deterministik per baris (tidak menghitung statistik
dari data), sehingga aman dipakai sebelum train/test split dan tidak menyebabkan
kebocoran data. Statistik yang dipelajari (encoding) dihitung di dalam pipeline
scikit-learn hanya dari data train.
"""
import pandas as pd

TARGET = "survived"
KOLOM_ASLI = ["class", "sex", "age"]
NILAI_VALID = {
    "class": ["1st", "2nd", "3rd", "Crew"],
    "sex": ["Male", "Female"],
    "age": ["Child", "Adult"],
    "survived": ["No", "Yes"],
}

# Kolom yang masuk model setelah feature engineering (semuanya kategorik)
FITUR_KATEGORIK = [
    "class", "sex", "age",
    "class_sex", "class_age", "sex_age", "class_sex_age",
    "wanita_atau_anak",
]


def muat_data(path):
    """Baca CSV, rapikan nama kolom/nilai, dan buang kolom indeks bawaan."""
    df = pd.read_csv(path)
    df.columns = [str(c).strip().lower() for c in df.columns]
    df = df.loc[:, ~df.columns.str.startswith("unnamed")]  # kolom indeks (jika ada)
    for kol in NILAI_VALID:
        if kol in df.columns:
            df[kol] = df[kol].astype(str).str.strip().str.title()
    # "Crew" dan "1st" tetap benar setelah title-case; pastikan konsisten
    df["class"] = df["class"].replace({"1St": "1st", "2Nd": "2nd", "3Rd": "3rd"})
    return df


def audit_data(df):
    """Audit data: duplikat, missing value, nilai tidak wajar, tipe data."""
    hasil = {
        "jumlah_baris": int(len(df)),
        "tipe_data": df.dtypes.astype(str).to_dict(),
        "missing_per_kolom": df.isna().sum().to_dict(),
        "baris_duplikat": int(df.duplicated().sum()),
        "kombinasi_unik": int(len(df.drop_duplicates())),
        "nilai_tidak_wajar": {},
    }
    for kol, valid in NILAI_VALID.items():
        if kol in df.columns:
            asing = sorted(set(df[kol].dropna().unique()) - set(valid))
            hasil["nilai_tidak_wajar"][kol] = asing
    return hasil


def tambah_fitur(X):
    """Buat fitur baru dari input mentah (class, sex, age).

    Dipakai sebagai langkah pertama pipeline, sehingga aplikasi Streamlit
    cukup mengirim input mentah.
    """
    X = X[KOLOM_ASLI].copy()
    for kol in KOLOM_ASLI:
        X[kol] = X[kol].astype(str)
    # Interaksi 2 arah dan 3 arah
    X["class_sex"] = X["class"] + "_" + X["sex"]
    X["class_age"] = X["class"] + "_" + X["age"]
    X["sex_age"] = X["sex"] + "_" + X["age"]
    X["class_sex_age"] = X["class"] + "_" + X["sex"] + "_" + X["age"]
    # Fitur domain: aturan evakuasi "women and children first"
    X["wanita_atau_anak"] = (
        (X["sex"] == "Female") | (X["age"] == "Child")
    ).map({True: "Ya", False: "Tidak"})
    return X[FITUR_KATEGORIK]


def siapkan_xy(df):
    """Pisahkan fitur mentah dan target biner (Yes=1, No=0)."""
    X = df[KOLOM_ASLI].copy()
    y = (df[TARGET] == "Yes").astype(int)
    return X, y


In [ ]:
%%writefile latih.py
"""latih.py - Pelatihan, evaluasi, dan penyimpanan model Titanic (Kasus 10).

Jalankan:  python latih.py
Opsional:  python latih.py --data dataset/kasus10_titanic.csv
"""
import argparse
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, f1_score, precision_score,
                             recall_score, roc_auc_score)
from sklearn.model_selection import (StratifiedKFold, cross_val_predict,
                                     cross_validate, train_test_split)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder

import fitur

BASE = Path(__file__).resolve().parent
SEED = 42


def buat_pipeline(model):
    """Pipeline lengkap: input mentah -> fitur baru -> one-hot -> model."""
    prep = ColumnTransformer(
        [("kat", OneHotEncoder(handle_unknown="ignore"), fitur.FITUR_KATEGORIK)]
    )
    return Pipeline([
        ("fitur", FunctionTransformer(fitur.tambah_fitur)),
        ("prep", prep),
        ("model", model),
    ])


def main(path_data):
    df = fitur.muat_data(path_data)

    # 1) AUDIT DATA
    audit = fitur.audit_data(df)
    print("=== AUDIT DATA ===")
    print(json.dumps(audit, indent=2, ensure_ascii=False))
    df = df.dropna(subset=fitur.KOLOM_ASLI + [fitur.TARGET])
    for kol, valid in fitur.NILAI_VALID.items():
        df = df[df[kol].isin(valid)]
    # Baris duplikat SENGAJA tidak dibuang: setiap baris adalah satu orang,
    # jadi duplikat membawa informasi frekuensi (berapa banyak orang per kombinasi).

    X, y = fitur.siapkan_xy(df)

    # 2) SPLIT DULU (sebelum encoding / statistik apa pun)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=SEED
    )
    print(f"\nTrain: {len(X_train)} | Test: {len(X_test)} | "
          f"Proporsi selamat train={y_train.mean():.3f}, test={y_test.mean():.3f}")

    # 3) BANDINGKAN MODEL DENGAN CROSS-VALIDATION DI DATA TRAIN
    kandidat = {
        "Logistic Regression": LogisticRegression(max_iter=1000, random_state=SEED),
        "Random Forest": RandomForestClassifier(
            n_estimators=300, min_samples_leaf=5, random_state=SEED),
        "Gradient Boosting": GradientBoostingClassifier(random_state=SEED),
    }
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    skor = ["accuracy", "precision", "recall", "f1", "roc_auc"]
    ringkasan_cv = {}
    print("\n=== CROSS-VALIDATION (5-fold, data train) ===")
    for nama, model in kandidat.items():
        hasil = cross_validate(buat_pipeline(model), X_train, y_train, cv=cv, scoring=skor)
        ringkasan_cv[nama] = {
            m: [round(float(hasil[f"test_{m}"].mean()), 4),
                round(float(hasil[f"test_{m}"].std()), 4)] for m in skor
        }
        r = ringkasan_cv[nama]
        print(f"{nama:20s} acc={r['accuracy'][0]:.3f} prec={r['precision'][0]:.3f} "
              f"rec={r['recall'][0]:.3f} f1={r['f1'][0]:.3f} auc={r['roc_auc'][0]:.3f}")

    # Pilih model dengan ROC-AUC CV tertinggi (tidak bergantung threshold).
    # Jika selisih AUC < 0.005 (praktis seri), pilih model yang lebih sederhana
    # (urutan kandidat: dari yang paling sederhana).
    auc_maks = max(r["roc_auc"][0] for r in ringkasan_cv.values())
    terbaik = next(n for n in kandidat if ringkasan_cv[n]["roc_auc"][0] >= auc_maks - 0.005)
    print(f"\nModel terpilih (ROC-AUC CV, tie -> paling sederhana): {terbaik}")

    # Tentukan threshold dari prediksi out-of-fold di data TRAIN (bukan data test).
    # Kelas 'selamat' minoritas (~32%), jadi threshold 0.5 membuat recall rendah.
    oof = cross_val_predict(buat_pipeline(kandidat[terbaik]), X_train, y_train,
                            cv=cv, method="predict_proba")[:, 1]
    grid = np.round(np.arange(0.20, 0.71, 0.01), 2)
    f1_grid = [f1_score(y_train, (oof >= t).astype(int)) for t in grid]
    threshold = float(grid[int(np.argmax(f1_grid))])
    print(f"Threshold terpilih dari OOF train (F1 maks): {threshold:.2f}")

    # 4) LATIH ULANG DI SELURUH TRAIN, EVALUASI TEST SATU KALI
    pipe = buat_pipeline(kandidat[terbaik]).fit(X_train, y_train)
    proba = pipe.predict_proba(X_test)[:, 1]
    pred = (proba >= threshold).astype(int)
    metrik = {
        "model": terbaik,
        "threshold": threshold,
        "accuracy": round(float(accuracy_score(y_test, pred)), 4),
        "precision": round(float(precision_score(y_test, pred)), 4),
        "recall": round(float(recall_score(y_test, pred)), 4),
        "f1": round(float(f1_score(y_test, pred)), 4),
        "roc_auc": round(float(roc_auc_score(y_test, proba)), 4),
        "baseline_mayoritas": round(float(1 - y_test.mean()), 4),
        "confusion_matrix": confusion_matrix(y_test, pred).tolist(),
        "n_train": int(len(X_train)),
        "n_test": int(len(X_test)),
    }
    print("\n=== EVALUASI DATA TEST (satu kali) ===")
    print(json.dumps(metrik, indent=2))
    print(classification_report(y_test, pred, target_names=["Tidak selamat", "Selamat"]))

    # 5) SIMPAN PIPELINE + METADATA
    (BASE / "models").mkdir(exist_ok=True)
    paket = {
        "pipeline": pipe,
        "metadata": {
            "fitur_input": fitur.KOLOM_ASLI,
            "pilihan": {k: v for k, v in fitur.NILAI_VALID.items() if k != "survived"},
            "label": {0: "Tidak selamat", 1: "Selamat"},
            "metrik_test": metrik,
            "cv": ringkasan_cv,
            "audit": audit,
        },
    }
    joblib.dump(paket, BASE / "models" / "titanic_pipeline.joblib")
    with open(BASE / "models" / "metrik.json", "w", encoding="utf-8") as f:
        json.dump({"test": metrik, "cv": ringkasan_cv, "audit": audit}, f,
                  indent=2, ensure_ascii=False)
    print("\nModel tersimpan di models/titanic_pipeline.joblib")
    tulis_laporan(audit, ringkasan_cv, metrik, threshold)


def tulis_laporan(audit, cv, m, threshold):
    """Hasilkan laporan.md (angka diambil langsung dari hasil run)."""
    (tn, fp), (fn, tp) = m["confusion_matrix"]
    baris_cv = "\n".join(
        f"| {n} | {r['accuracy'][0]:.3f} | {r['f1'][0]:.3f} | {r['roc_auc'][0]:.3f} |"
        for n, r in cv.items())
    teks = f"""# Laporan UTS Intelegensi Buatan - Kasus 10 (Titanic)

Nama: Rizal Fahatrik Nugraha | NIM: 23552011452 | Kelas: TIF RP 23D

## 1. Feature engineering
- **Audit data**: {audit['jumlah_baris']} baris, missing value = {sum(audit['missing_per_kolom'].values())}, tidak ada nilai di luar kategori valid. Tipe data semua kolom teks (kategorik). Ada {audit['baris_duplikat']} baris duplikat karena data hanya punya 3 fitur kategorik (maksimal 16 kombinasi). Duplikat **tidak dibuang**: tiap baris adalah satu orang, sehingga duplikat membawa informasi frekuensi.
- **Kolom dipakai**: class, sex, age. Tidak ada kolom ID atau kolom yang baru diketahui setelah kejadian. `survived` adalah target.
- **Fitur baru**: interaksi `class_sex`, `class_age`, `sex_age`, `class_sex_age`, serta `wanita_atau_anak` (aturan evakuasi "women and children first"). Interaksi membantu model linear (Logistic Regression) karena model ini bersifat aditif dan tidak menangkap interaksi sendiri; mis. efek jenis kelamin sangat berbeda antar kelas.
- **Encoding**: One-Hot Encoding (`handle_unknown="ignore"`) untuk semua fitur. Kelas penumpang tidak diberi ordinal karena kategori `Crew` bukan bagian dari urutan 1st-2nd-3rd.
- **Tanpa kebocoran data**: data dibagi 80:20 (stratified) sebelum encoding. Encoder dipelajari hanya dari data train di dalam pipeline.

## 2. Perbandingan model (5-fold CV, data train)
| Model | Accuracy | F1 | ROC-AUC |
|---|---|---|---|
{baris_cv}

Ketiga model hampir identik karena data hanya memiliki sedikit kombinasi unik, sehingga model yang kompleks tidak punya informasi tambahan untuk dipelajari. Dipilih **{m['model']}** (ROC-AUC tertinggi; jika selisih < 0,005 dipilih yang paling sederhana dan mudah diinterpretasi).

## 3. Hasil pada data test (dievaluasi satu kali)
Threshold = {threshold:.2f}, ditentukan dari prediksi out-of-fold pada data train (bukan data test), karena kelas "selamat" minoritas.

| Accuracy | Precision | Recall | F1 | ROC-AUC | Baseline mayoritas |
|---|---|---|---|---|---|
| {m['accuracy']:.3f} | {m['precision']:.3f} | {m['recall']:.3f} | {m['f1']:.3f} | {m['roc_auc']:.3f} | {m['baseline_mayoritas']:.3f} |

Confusion matrix: TN={tn}, FP={fp}, FN={fn}, TP={tp}.

## 4. Interpretasi
- **Accuracy** hanya dibandingkan dengan baseline (selalu menebak "tidak selamat") karena kelas tidak seimbang; selisihnya menunjukkan nilai tambah model.
- **Precision** {m['precision']:.2f}: dari semua yang diprediksi selamat, sebesar itu yang benar-benar selamat. **Recall** {m['recall']:.2f}: dari semua penyintas nyata, sebesar itu yang berhasil terdeteksi.
- **F1** menyeimbangkan precision dan recall. **ROC-AUC** {m['roc_auc']:.2f} mengukur kemampuan model mengurutkan penumpang dari yang paling mungkin selamat, tanpa bergantung pada threshold.
- Threshold 0,50 menghasilkan recall rendah; menurunkannya menaikkan recall dengan sedikit mengorbankan precision.
- **Keterbatasan**: hanya 3 fitur kategorik, sehingga banyak penumpang dengan profil sama punya nasib berbeda. Batas atas performa model rendah dan itu bukan kesalahan modeling. Model menggambarkan pola historis, bukan kepastian individu.
- **Deploy**: pipeline lengkap + metadata disimpan di `models/titanic_pipeline.joblib`; aplikasi Streamlit (`app.py`) menerima input mentah (class, sex, age).
"""
    (BASE / "laporan.md").write_text(teks, encoding="utf-8")
    print("Laporan tertulis di laporan.md")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default=str(BASE / "dataset" / "kasus10_titanic.csv"))
    main(ap.parse_args().data)


In [ ]:
%%writefile app.py
"""app.py - Aplikasi Streamlit prediksi keselamatan penumpang Titanic.

Jalankan:  streamlit run app.py
"""
from pathlib import Path

import joblib
import pandas as pd
import streamlit as st

import fitur  # dibutuhkan agar pipeline joblib dapat dimuat

BASE = Path(__file__).resolve().parent
PATH_MODEL = BASE / "models" / "titanic_pipeline.joblib"

st.set_page_config(page_title="Prediksi Penumpang Titanic", page_icon="🚢")
st.title("🚢 Prediksi Keselamatan Penumpang Titanic")


@st.cache_resource
def muat_model():
    return joblib.load(PATH_MODEL)


if not PATH_MODEL.exists():
    st.error("Model belum ada. Jalankan `python latih.py` terlebih dahulu.")
    st.stop()

paket = muat_model()
pipe, meta = paket["pipeline"], paket["metadata"]
pilihan = meta["pilihan"]
nama_kelas = {"1st": "Kelas 1", "2nd": "Kelas 2", "3rd": "Kelas 3", "Crew": "Awak kapal"}
nama_sex = {"Male": "Laki-laki", "Female": "Perempuan"}
nama_age = {"Child": "Anak-anak", "Adult": "Dewasa"}

tab_prediksi, tab_metrik = st.tabs(["Prediksi", "Evaluasi model"])

with tab_prediksi:
    st.subheader("Masukkan data penumpang")
    c1, c2, c3 = st.columns(3)
    kelas = c1.selectbox("Kelas", pilihan["class"], format_func=nama_kelas.get)
    sex = c2.selectbox("Jenis kelamin", pilihan["sex"], format_func=nama_sex.get)
    age = c3.selectbox("Usia", pilihan["age"], format_func=nama_age.get)

    if st.button("Prediksi", type="primary"):
        X = pd.DataFrame([{"class": kelas, "sex": sex, "age": age}])
        proba = pipe.predict_proba(X)[0]
        p_selamat = float(proba[list(pipe.classes_).index(1)])
        ambang = meta["metrik_test"]["threshold"]
        if p_selamat >= ambang:
            st.success(f"Prediksi: **SELAMAT** (probabilitas {p_selamat:.1%})")
        else:
            st.error(f"Prediksi: **TIDAK SELAMAT** (probabilitas selamat {p_selamat:.1%})")
        st.write("Probabilitas tiap kelas:")
        st.bar_chart(pd.DataFrame(
            {"Probabilitas": [1 - p_selamat, p_selamat]},
            index=["Tidak selamat", "Selamat"]))
        st.caption("Data historis 1912; hasil menggambarkan pola data, bukan kepastian individu.")

with tab_metrik:
    m = meta["metrik_test"]
    st.subheader(f"Model terpilih: {m['model']}")
    st.caption(f"Dievaluasi sekali pada data test ({m['n_test']} baris), threshold {m['threshold']}")
    k = st.columns(5)
    k[0].metric("Accuracy", f"{m['accuracy']:.3f}")
    k[1].metric("Precision", f"{m['precision']:.3f}")
    k[2].metric("Recall", f"{m['recall']:.3f}")
    k[3].metric("F1", f"{m['f1']:.3f}")
    k[4].metric("ROC-AUC", f"{m['roc_auc']:.3f}")
    st.write(f"Baseline (selalu tebak 'Tidak selamat'): accuracy {m['baseline_mayoritas']:.3f}")
    st.write("Confusion matrix (baris = aktual, kolom = prediksi):")
    st.dataframe(pd.DataFrame(
        m["confusion_matrix"],
        index=["Aktual: Tidak selamat", "Aktual: Selamat"],
        columns=["Prediksi: Tidak selamat", "Prediksi: Selamat"]))
    st.write("Perbandingan model (cross-validation 5-fold pada data train, rata-rata ± std):")
    baris = []
    for nama, r in meta["cv"].items():
        baris.append({"Model": nama, **{mt: f"{v[0]:.3f} ± {v[1]:.3f}" for mt, v in r.items()}})
    st.dataframe(pd.DataFrame(baris), hide_index=True)


## 2. Dataset
Unggah `kasus10_titanic.csv` (atau pakai unduhan otomatis dari GitHub jika tersedia).

In [1]:
import os
PATH = 'dataset/kasus10_titanic.csv'
if not os.path.exists(PATH):
    try:
        import urllib.request
        URL = 'https://raw.githubusercontent.com/Muhammad-Ikhwan-Fathulloh/ArtificialIntelegent-Course-Bank/main/UTS/dataset/kasus10_titanic.csv'
        urllib.request.urlretrieve(URL, PATH)
        print('Diunduh dari GitHub')
    except Exception as e:
        print('Unduh otomatis gagal, silakan unggah manual:', e)
        from google.colab import files
        up = files.upload()
        os.replace(list(up.keys())[0], PATH)
import pandas as pd
pd.read_csv(PATH).head()

Unduh otomatis gagal, silakan unggah manual: HTTP Error 404: Not Found


ModuleNotFoundError: No module named 'google.colab'

## 3. Training, evaluasi, dan simpan model (joblib)

In [ ]:
!python latih.py --data dataset/kasus10_titanic.csv

## 4. Laporan otomatis

In [ ]:
print(open('laporan.md', encoding='utf-8').read())

## 5. Jalankan aplikasi Streamlit di Colab
Sel berikut memasang Streamlit, menjalankan `app.py`, lalu membuka tunnel. Buka URL `https://....loca.lt` yang muncul; jika diminta password, isi dengan IP yang ditampilkan.

In [ ]:
!pip install -q streamlit

In [ ]:
!npm install -q localtunnel
!streamlit run app.py --server.port 8501 --server.headless true &>/content/logs.txt &
import time; time.sleep(5)
print('IP (password tunnel):')
!curl -s ipv4.icanhazip.com
!npx localtunnel --port 8501